In [1]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform

import duckdb
import pandas as pd

# ==========================
# 1. Import
# ==========================
import numpy as np


# ==========================
# 2. Parâmetros
# ==========================
GESTAO = "LIS"

ANOS = [
    2025,
    2026,
]


# ==========================
# 3. Pasta das bases
# ==========================
def get_db_dir():

    sistema = platform.system()

    if sistema == "Windows":
        return Path(
            r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB"
        )

    if sistema == "Darwin":
        return Path(
            "/Users/rr/Library/Mobile Documents/"
            "com~apple~CloudDocs/05.Salvesen/00_DB"
        )

    return Path.cwd()


DB_DIR = get_db_dir()


# ==========================
# 4. Caminhos
# ==========================
PATHS = {
    ano: DB_DIR / f"{ano}.duckdb"
    for ano in ANOS
}


# ==========================
# 5. Validar bases
# ==========================
for ano, path in PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Base {ano} não encontrada: {path}"
        )


# ==========================
# 6. Ligações DuckDB
# ==========================
CON = {
    ano: duckdb.connect(
        str(PATHS[ano]),
        read_only=True
    )
    for ano in ANOS
}


# ==========================
# 7. Validar
# ==========================
validacao = pd.DataFrame({
    "Ano": ANOS,
    "Base": [
        PATHS[ano].name
        for ano in ANOS
    ],
    "Tabelas": [
        len(CON[ano].sql("SHOW TABLES").fetchall())
        for ano in ANOS
    ],
})

validacao


,Ano,Base,Tabelas
0,2025,2025.duckdb,8
1,2026,2026.duckdb,11


In [2]:
# ==========================
# ENTREGAS MACHADO E BRITES
# Ano completo - sem filtro de mês
# ==========================

ANO = 2026

df_entregas = CON[ANO].execute(
    f"""
    SELECT DISTINCT
        FENTREGA,
        CODEUT

    FROM inform_27_{ANO}

    WHERE Gestion = ?
      AND TRANSPORTISTA ILIKE '%MACHADO%'
      AND TRANSPORTISTA ILIKE '%BRITES%'

    ORDER BY
        FENTREGA,
        CODEUT
    """,
    [GESTAO]
).df()

# Visualizar resultados
print(f"ANO: {ANO}")
print(f"TOTAL DE ENTREGAS: {len(df_entregas)}")

with pd.option_context("display.max_rows", None):
    display(df_entregas)

ANO: 2026
TOTAL DE ENTREGAS: 15


,FENTREGA,CODEUT
0,20260806,3514019
1,20260807,3514775
2,20260810,3515749
3,20260811,3517262
4,20260811,3518568
5,20260812,3518143
6,20260812,3519571
7,20260813,3519221
8,20260814,3520565
9,20260817,3521477


In [3]:
# ==========================================
# CUSTOS, INGRESSOS, PALETES E PESO BRUTO
# POR REFERENCIA - LINHA A LINHA
# Agosto e Setembro de 2026
# ==========================================

ANO = 2026

CLIENTES = [
    '391', '404', '405', '407', '415',
    '551', '680', '757', '311'
]

df_clientes = CON[ANO].execute(
    f"""
    SELECT
        SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) AS MES,

        FENTREGA,

        LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0') AS CODACT,

        ACTIVIDAD,

        REFERENCIA,

        ROUND(
            TRY_CAST(REPLACE(CAST(INGRESODT AS VARCHAR), ',', '.') AS DOUBLE),
            2
        ) AS INGRESODT,

        ROUND(
            TRY_CAST(REPLACE(CAST(COSTEDT AS VARCHAR), ',', '.') AS DOUBLE),
            2
        ) AS COSTEDT,

        ROUND(
            TRY_CAST(REPLACE(CAST(PALETS AS VARCHAR), ',', '.') AS DOUBLE),
            2
        ) AS PALETS,

        ROUND(
            TRY_CAST(REPLACE(CAST(PESO_BRUTO AS VARCHAR), ',', '.') AS DOUBLE),
            2
        ) AS PESO_BRUTO

    FROM inform_27_{ANO}

    WHERE Gestion = ?

      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6)
          IN ('202608', '202609')

      AND LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0')
          IN ({','.join(['?'] * len(CLIENTES))})

    ORDER BY
        CODACT,
        MES,
        REFERENCIA
    """,
    [GESTAO, *CLIENTES]
).df()

# Visualizar todas as linhas
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(df_clientes)

print(f"Total de linhas: {len(df_clientes):,}")

,MES,FENTREGA,CODACT,ACTIVIDAD,REFERENCIA,INGRESODT,COSTEDT,PALETS,PESO_BRUTO
0,202608,20260803,311,ALPRO PORTUGAL,5023384467 415611177,0.00,0.53,1.0,29.96
1,202608,20260804,311,ALPRO PORTUGAL,5023387357 415583821,0.00,40.10,1.0,233.46
2,202608,20260804,311,ALPRO PORTUGAL,5023387358 415583822,0.00,48.40,1.0,389.10
3,202608,20260812,311,ALPRO PORTUGAL,5023393113 415682682,6.64,0.60,1.0,31.69
4,202608,20260812,311,ALPRO PORTUGAL,5023393114 415677556,0.00,3.79,1.0,12.96
5,202608,20260804,311,ALPRO PORTUGAL,5023398834 415590092,0.00,22.18,1.0,181.58
6,202608,20260807,311,ALPRO PORTUGAL,5023401427 415626635,0.00,1.50,1.0,72.41
7,202608,20260817,311,ALPRO PORTUGAL,5023401438 415705975,13.53,5.09,1.0,49.42
8,202608,20260803,311,ALPRO PORTUGAL,5023409230 415555928,0.00,37.04,2.0,1621.37
9,202608,20260803,311,ALPRO PORTUGAL,5023412267 415604646,0.00,22.90,2.0,1127.80


Total de linhas: 7,944


In [4]:
# ==========================
# EXPORTAR PARA EXCEL
# ==========================

from pathlib import Path

ficheiro = (
    Path.home()
    / "Documents"
    / "Clientes_Referencias_Agosto_Setembro_2026.xlsx"
)

df_clientes.to_excel(ficheiro, index=False)

print(f"Excel exportado: {ficheiro}")
print(f"Total de linhas exportadas: {len(df_clientes):,}")

Excel exportado: C:\Users\LISARR\Documents\Clientes_Referencias_Agosto_Setembro_2026.xlsx
Total de linhas exportadas: 7,944
